<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_4_4_MLR_Ames_Part4_Revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLR Predicting Housing Prices in Ames Iowa: Part 4
## Hyperparameter Tuning with Grid Search

Author: Brad Sheese

**Recap:**
*   In Part 3 we introduced Regularization (Ridge, Lasso, ElasticNet). We learned that by penalizing large coefficients, we can stabilize our models and prevent overfitting.
*   However, in Part 3, we arbitrarily picked the penalty strength ($\alpha=1.0$ or $\alpha=0.01$). We guessed.

**This Notebook:**
*   We stop guessing. We will use Cross-Validation and Grid Search to find the optimal hyperparameters for our models.
*   We will visualize the Bias-Variance Tradeoff to understand why a specific alpha value performs best.

**Data Source:** http://jse.amstat.org/v19n3/decock/AmesHousing.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Download the cleaning module from GitHub
import urllib.request
module_url = "https://raw.githubusercontent.com/bsheese/cs377/main/17_regression_crossval/ames_cleaning.py"
urllib.request.urlretrieve(module_url, "ames_cleaning.py")
from ames_cleaning import load_and_clean_ames

# Load and clean the Ames dataset
df = load_and_clean_ames(one_hot_encode=True)
print(f"DataFrame shape: {df.shape}")


In [ ]:
from sklearn.model_selection import train_test_split

# Define features and target
# Remember: We log-transform the target to handle skewness
X = df.drop(columns=['SalePrice'])
y = df['SalePrice'].map(np.log)

# Split the data (80% Train, 20% Test)
# We will perform Cross-Validation ONLY on the Training set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Why Tune Hyperparameters?

In regularized regression (Ridge, Lasso, ElasticNet), the **Alpha ($\alpha$)** parameter controls the strength of the penalty.

*   **Too Low ($\alpha \approx 0$)**: The model behaves like OLS. It might overfit the training data (High Variance).
*   **Too High ($\alpha \rightarrow \infty$)**: The model is penalized too heavily. Coefficients are crushed to zero. It underfits the data (High Bias).

We need to find the "Goldilocks" zone: not too hot, not too cold.

## 1. Visualizing the Bias-Variance Tradeoff (Manual Tuning)

Before automating this, let's manually loop through a list of possible alpha values for **Ridge Regression** and see how the error changes.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Range of alphas to test (logarithmic scale is usually best)
alphas = [0.01, 0.1, 1, 10, 100, 1000]

cv_scores = []
train_scores = []

for alpha in alphas:
    # Create pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=alpha))
    ])

    # Cross-validation score (R-squared)
    # Note: cross_val_score returns the score for each fold
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
    cv_scores.append(scores.mean())

    # Also fit on full training set to check training score (for comparison)
    pipeline.fit(X_train, y_train)
    train_scores.append(pipeline.score(X_train, y_train))

# Plotting
plt.figure(figsize=(10, 6))
plt.semilogx(alphas, train_scores, label='Training R² (Higher is better)', marker='o')
plt.semilogx(alphas, cv_scores, label='Cross-Val R² (Higher is better)', marker='o')
plt.xlabel('Alpha (Penalty Strength)')
plt.ylabel('R-Squared Score')
plt.title('Ridge Regression: Bias-Variance Tradeoff')
plt.legend()
plt.grid(True)
plt.show()

**Interpretation:**
*   **Low Alpha:** Training score is high, but CV score is slightly lower (potential overfitting).
*   **High Alpha:** Both scores drop significantly (underfitting - the model is too simple).
*   **The Peak:** The alpha where the CV score is maximized is our optimal hyperparameter.

## 2. Automated Tuning with GridSearchCV

Instead of writing loops manually, Scikit-Learn provides `GridSearchCV`.

It takes:
1.  **Estimator** (your model or pipeline)
2.  **Param Grid** (dictionary of parameters to test)
3.  **CV** (number of folds)

It runs every combination and finds the best one.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def tune_model(model, param_grid, X_train, y_train):
    """
    Performs GridSearchCV to find the best hyperparameters.

    Why a Pipeline?
    We must scale the data *inside* the cross-validation loop. If we scaled the
    entire dataset beforehand, information from the validation fold would 'leak'
    into the training fold (via the mean and standard deviation). The Pipeline
    ensures that for every fold, the scaler is fit ONLY on the training data.
    """
    # 1. Create the pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    # 2. Rename params for the pipeline
    # The pipeline steps are named 'scaler' and 'model'.
    # We must prefix our parameters with 'model__' so GridSearchCV knows where they go.
    pipeline_param_grid = {f'model__{key}': value for key, value in param_grid.items()}

    # 3. Setup Grid Search
    # refit=True (default) means: "After finding the best params, fit the model ONE MORE TIME
    # on the entire X_train/y_train so it is ready to use."
    grid_search = GridSearchCV(
        pipeline,
        pipeline_param_grid,
        cv=5,
        scoring='r2',
        n_jobs=-1,
        verbose=1,
        refit=True # this is the default, and unnecessary, but I want you to see it).
    )

    # 4. Run the Search
    grid_search.fit(X_train, y_train)

    print(f"Best Parameters: {grid_search.best_params_}")
    print(f"Best CV R²:      {grid_search.best_score_:.4f}")

    # Return the fully fitted, best version of the pipeline
    return grid_search.best_estimator_

### 2A. Tuning Ridge Regression
We search for the best `alpha`.

In [ ]:
ridge_params = {'alpha': [0.01, 0.1, 1, 5, 10, 20, 50, 100]}
best_ridge = tune_model(Ridge(), ridge_params, X_train, y_train)

### 2B. Tuning Lasso Regression
Lasso is more sensitive to alpha. If alpha is too high, it drops ALL features.

In [ ]:
lasso_params = {'alpha': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 1]}
best_lasso = tune_model(Lasso(max_iter=5000), lasso_params, X_train, y_train)

### 2C. Tuning ElasticNet (2D Grid Search)
Here we search two dimensions:
1.  `alpha`: The overall penalty strength.
2.  `l1_ratio`: The mix. 0.0=Ridge, 1.0=Lasso.

GridSearchCV will try EVERY combination (Cartesian product).

In [ ]:
enet_params = {
    'alpha': [0.001, 0.01, 0.1, 1],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}
best_enet = tune_model(ElasticNet(max_iter=5000), enet_params, X_train, y_train)

## 3. Final Evaluation on Test Set

Now that we have tuned our models on the training set, let's see which one actually performs best on the Test Set that none of the models have seen yet.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

models = {
    'Optimized Ridge': best_ridge,
    'Optimized Lasso': best_lasso,
    'Optimized ElasticNet': best_enet
}

results = []

for name, model in models.items():
    # Predict
    y_pred = model.predict(X_test)

    # Score
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    # Count features (non-zero coefficients)
    # Steps: Pipeline -> 'model' step -> coef_
    coefs = model.named_steps['model'].coef_
    n_features = np.sum(np.abs(coefs) > 1e-5) # Count non-zero coefs

    results.append({
        'Model': name,
        'Test R²': f"{r2:.4f}",
        'Test MSE': f"{mse:.4f}",
        'Features Kept': n_features
    })

# Create comparison table
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

### Why Are Our Test Scores "Optimistic"?

Here's the problem: When we use GridSearchCV to find the *best* alpha, we're essentially "fitting to the test set" - but the test set here is our validation fold within cross-validation.

Think of it this way: We tried many alpha values (0.01, 0.1, 1, 10, 100, 1000) and picked the one that performed best on the validation fold. That "best alpha" is then applied to our test set. But there's a small chance that alpha=10 was just lucky on that particular validation fold - maybe alpha=50 would actually be better on truly new data.

**This is the same problem we solved for features in Part 2, now applied to hyperparameters!**

---

## Enter Nested Cross-Validation

Nested CV solves this by having **two loops**:

**🌀 Inner Loop (Tuning):**
- Within each training fold, GridSearchCV tries many alpha values
- It finds the "best" alpha for that specific training subset
- This is what we've been doing above

**🔄 Outer Loop (Evaluation):**
- The outer loop tests the model on data it's never seen
- It uses the best hyperparameters found by the *inner* loop
- This gives an *unbiased* estimate of performance

**Visual Example:**
```
Outer Fold 1: Train on A,B,C,D → Test on E
  Inner: Try alpha=0.1,1,10 → Best is α=1 ( picked from this training data)
  → Test on E using α=1

Outer Fold 2: Train on A,B,C,E → Test on D
  Inner: Try alpha=0.1,1,10 → Best is α=10 ( picked from this training data)
  → Test on D using α=10

Outer Fold 3: Train on A,B,D,E → Test on C
  Inner: Try alpha=0.1,1,10 → Best is α=0.1 ( picked from this training data)
  → Test on C using α=0.1
...

Final Score = Average of (E's score, D's score, C's score, ...)
```

The key insight: At test time, the model uses the alpha that was *tuned on training data*, not the one we guessed or GridSearch'd on that same data.

---

**Note:** Nested CV is computationally expensive (we're running CV inside CV). For quick iterations, we often accept the simpler "tuned on training, tested on test" approach above. But for production models or final evaluations, nested CV gives us a more honest estimate.

In [ ]:
from sklearn.model_selection import cross_val_score

# Nested Cross-Validation for unbiased performance estimate
# Note: This is computationally expensive - we're running 5x5 = 25 model fits!

print("Nested Cross-Validation (Unbiased Estimate)")
print("=" * 50)

models = {
    'Ridge (α=tuned)': Ridge(alpha=1.0),   # Using best alpha from above
    'Lasso (α=tuned)': Lasso(alpha=0.001, max_iter=10000),
    'ElasticNet (α=tuned)': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000)
}

# This IS the outer loop - giving us unbiased test estimates
for name, model in models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    
    cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='r2', n_jobs=-1)
    print(f"{name}: Nested CV R² = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print("\n--- Comparison ---")
print("Test Set R² (optimistic, tuned on same validation): ~0.88")
print("Nested CV R² (unbiased, truly out-of-sample): See above")
print("\nThe nested scores are typically LOWER because they account for")
print("the uncertainty in choosing the best hyperparameters.")

In [ ]:
from sklearn.model_selection import learning_curve

# Learning Curves - diagnose if model needs more data
best_ridge_pipeline = best_ridge

train_sizes_abs, train_scores_lc, test_scores_lc = learning_curve(
    best_ridge_pipeline,
    X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 5),
    cv=5,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

train_mean = train_scores_lc.mean(axis=1)
train_std = train_scores_lc.std(axis=1)
test_mean = test_scores_lc.mean(axis=1)
test_std = test_scores_lc.std(axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes_abs, train_mean, 'o-', color='blue', label='Training Score')
plt.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
plt.plot(train_sizes_abs, test_mean, 'o-', color='orange', label='Validation Score')
plt.fill_between(train_sizes_abs, test_mean - test_std, test_mean + test_std, alpha=0.1, color='orange')
plt.xlabel('Training Set Size')
plt.ylabel('R² Score')
plt.title('Learning Curves - Optimized Ridge')
plt.legend(loc='best')
plt.grid(True)
plt.ylim(-0.5, 1.0)
plt.show()

print("\n**Learning Curve Interpretation:**")
print("- If validation score is much lower than training score, the model is overfitting")
print("- If both scores converge at a low value, the model is underfitting (high bias)")
print("- If validation score is still improving with more data, we need more training examples")
print(f"- Current: Training R² = {train_mean[-1]:.3f}, Validation R² = {test_mean[-1]:.3f}")

## Conclusion

We have successfully moved from "guessing" to "optimizing." You will notice that the parameters used for the search in that last exercise were close to the optimal parameters we found here. I wanted to present a best case outcome in the previous exercise so I had previously conduced a grid search to find the values.

Moving forward, assumming you've got the time to run the analyses and can pay for the compute to do so, you will most likely always conduct a grid search instead of just guessing parameters.
